## Contexte du Projet
======================================================================================================================


L'équipe marketing a besoin d'aide pour un nouveau projet. Après avoir mené une étude auprès des utilisateurs, elle a constaté que 70 % d'entre eux, qui prévoient un voyage, souhaiteraient obtenir davantage d'informations sur leur destination.


De plus, les études auprès des utilisateurs montrent que les gens ont tendance à être sceptiques quant aux informations qu'ils lisent s'ils ne connaissent pas la marque qui a produit le contenu.


L'équipe marketing de Kayak souhaite donc créer une application qui recommandera aux utilisateurs des destinations pour leurs prochaines vacances. Cette application devra s'appuyer sur des données réelles concernant :


- Météo
- Hôtels dans la région

L'application devrait alors être en mesure de recommander les meilleures destinations et les meilleurs hôtels en fonction des variables ci-dessus à tout moment.

**Objectifs**

Le projet venant de démarrer, votre équipe ne dispose d'aucune donnée permettant de créer cette application. Votre tâche consistera donc à :

- Récupérer des données à partir de destinations **(Pour avoir le latitude et longitude de la ville)**
- Obtenez des données météorologiques pour chaque destination **(Necessite le lat et long recuperés à partir de Nominatim)**
- Obtenez des informations sur les hôtels pour chaque destination.
- Stockez toutes les informations ci-dessus dans un lac de données.
- Extrayez, transformez et chargez les données nettoyées de votre lac de données vers un entrepôt de données.

### A- Etape 1: Récuperation des coordonnées GPS à partir des déstinations
=============================================================================================================

Utilisez https://nominatim.org/ pour obtenir les coordonnées GPS de toutes les ville

In [ ]:
import time
from pathlib import Path

import pandas as pd
import requests

# UserAgent
USER_AGENT = "jedha-kayak/1.0 (henintsoa@nexthope.net)"

# Dossiers de sortie
RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

#print("pandas", pd.__version__, "| requests", requests.__version__)

In [14]:
url = "https://nominatim.openstreetmap.org/search"
params = {"q": "Colmar, France", "format": "json", "limit": 1}
headers = {"User-Agent": USER_AGENT}

response = requests.get(url, params=params, headers=headers, timeout=10)

print("Statut HTTP :", response.status_code)
print("URL réellement appelée :", response.url)
response.json()

Statut HTTP : 200
URL réellement appelée : https://nominatim.openstreetmap.org/search?q=Colmar%2C+France&format=json&limit=1


[{'place_id': 121164431,
  'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright',
  'osm_type': 'relation',
  'osm_id': 61423,
  'lat': '48.0777517',
  'lon': '7.3579641',
  'class': 'boundary',
  'type': 'administrative',
  'place_rank': 16,
  'importance': 0.6305771963242904,
  'addresstype': 'town',
  'name': 'Colmar',
  'display_name': "Colmar, Colmar-Ribeauvillé, Haut-Rhin, Collectivité européenne d'Alsace, Grand Est, France métropolitaine, 68000, France",
  'boundingbox': ['48.0407053', '48.1820620', '7.3154803', '7.4691670']}]

In [ ]:
class Geocoder:
    def __init__(self, user_agent):
        # ses "attributs
        self.url = "https://nominatim.openstreetmap.org/search"
        self.headers = {"User-Agent": user_agent}

    def geocode(self, city):        
        params = {"q": f"{city}, France", "format": "json",
                  "limit": 1, "countrycodes": "fr"}
        response = requests.get(self.url, params=params,
                                headers=self.headers, timeout=10)
        results = response.json()

        if not results:
            return None

        top = results[0]
        return {
            "city": city,
            "lat": float(top["lat"]),
            "lon": float(top["lon"]),
            "address_type": top.get("addresstype"),
        }

In [16]:
geo = Geocoder(user_agent=USER_AGENT)

geo.geocode("Colmar")


{'city': 'Colmar', 'lat': 48.0777517, 'lon': 7.3579641, 'address_type': 'town'}

In [18]:
CITIES = [
    "Mont Saint Michel", "St Malo", "Bayeux", "Le Havre", "Rouen",
    "Paris", "Amiens", "Lille", "Strasbourg", "Chateau du Haut Koenigsbourg",
    "Colmar", "Eguisheim", "Besancon", "Dijon", "Annecy",
    "Grenoble", "Lyon", "Gorges du Verdon", "Bormes les Mimosas", "Cassis",
    "Marseille", "Aix en Provence", "Avignon", "Uzes", "Nimes",
    "Aigues Mortes", "Saintes Maries de la mer", "Collioure", "Carcassonne", "Ariege",
    "Toulouse", "Montauban", "Biarritz", "Bayonne", "La Rochelle",
]

rows = []
echecs = []

for city_id, city in enumerate(CITIES, start=1):
    try:
        result = geo.geocode(city)
    except requests.exceptions.RequestException as e:
        # Panne réseau, timeout, serveur injoignable : on note et on continue
        print(f"[ÉCHEC RÉSEAU] {city} : {e}")
        echecs.append(city)
        time.sleep(2)
        continue

    if result is None:
        print(f"[INTROUVABLE] {city}")
        result = {"city": city, "lat": None, "lon": None, "address_type": None}

    result["city_id"] = city_id
    rows.append(result)
    time.sleep(1)

print(f"\nRécupérées : {len(rows)} | Échecs réseau : {echecs}")

[ÉCHEC RÉSEAU] Chateau du Haut Koenigsbourg : HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Max retries exceeded with url: /search?q=Chateau+du+Haut+Koenigsbourg%2C+France&format=json&limit=1&countrycodes=fr (Caused by NewConnectionError("HTTPSConnection(host='nominatim.openstreetmap.org', port=443): Failed to establish a new connection: [Errno 101] Network is unreachable"))
[ÉCHEC RÉSEAU] Cassis : HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Read timed out. (read timeout=10)

Récupérées : 33 | Échecs réseau : ['Chateau du Haut Koenigsbourg', 'Cassis']


In [ ]:


restants = echecs          
echecs = []                


for city in restants:
    city_id = CITIES.index(city) + 1     

    try:
        result = geo.geocode(city)
    except requests.exceptions.RequestException as e:
        print(f"[ENCORE ÉCHEC] {city}")
        echecs.append(city)
        time.sleep(2)
        continue

    if result is None:
        print(f"[INTROUVABLE] {city}")
        result = {"city": city, "lat": None, "lon": None, "address_type": None}

    result["city_id"] = city_id
    rows.append(result)                  # on AJOUTE aux 31 déjà présentes
    print(f"[RÉCUPÉRÉE] {city}")
    time.sleep(1)

print(f"\nTotal maintenant : {len(rows)} villes | Restent en échec : {echecs}")

print(rows)

[RÉCUPÉRÉE] Chateau du Haut Koenigsbourg
[RÉCUPÉRÉE] Cassis

Total maintenant : 35 villes | Restent en échec : []
[{'city': 'Mont Saint Michel', 'lat': 48.6359541, 'lon': -1.51146, 'address_type': 'islet', 'city_id': 1}, {'city': 'St Malo', 'lat': 48.649518, 'lon': -2.0260409, 'address_type': 'town', 'city_id': 2}, {'city': 'Bayeux', 'lat': 49.2764624, 'lon': -0.7024738, 'address_type': 'town', 'city_id': 3}, {'city': 'Le Havre', 'lat': 49.4938975, 'lon': 0.1079732, 'address_type': 'city', 'city_id': 4}, {'city': 'Rouen', 'lat': 49.4404591, 'lon': 1.0939658, 'address_type': 'city', 'city_id': 5}, {'city': 'Paris', 'lat': 48.8534951, 'lon': 2.3483915, 'address_type': 'city', 'city_id': 6}, {'city': 'Amiens', 'lat': 49.8941708, 'lon': 2.2956951, 'address_type': 'city', 'city_id': 7}, {'city': 'Lille', 'lat': 50.6365654, 'lon': 3.0635282, 'address_type': 'city', 'city_id': 8}, {'city': 'Strasbourg', 'lat': 48.584614, 'lon': 7.7507127, 'address_type': 'city', 'city_id': 9}, {'city': 'Colma

### Sauvegardons dans un dataframe

In [20]:
df = pd.DataFrame(rows)
df = df[["city_id", "city", "lat", "lon", "address_type"]]
df

,city_id,city,lat,lon,address_type
0,1,Mont Saint Michel,48.635954,-1.511460,islet
1,2,St Malo,48.649518,-2.026041,town
2,3,Bayeux,49.276462,-0.702474,town
3,4,Le Havre,49.493898,0.107973,city
4,5,Rouen,49.440459,1.093966,city
5,6,Paris,48.853495,2.348391,city
6,7,Amiens,49.894171,2.295695,city
7,8,Lille,50.636565,3.063528,city
8,9,Strasbourg,48.584614,7.750713,city
9,11,Colmar,48.077752,7.357964,town


In [21]:
## Sauvegardons dans un csv pour ne pas appeler à chaque fois
df.to_csv(RAW_DIR / "cities.csv", index=False)
print("Écrit :", RAW_DIR / "cities.csv")

Écrit : data/raw/cities.csv


In [ ]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("data/raw/cities.csv")


df_map = df.dropna(subset=["lat", "lon"])

In [ ]:
fig = px.scatter_map(
    df_map,
    lat="lat",                    # colonne des latitudes
    lon="lon",                    # colonne des longitudes
    hover_name="city",            # titre de l'infobulle au survol
    hover_data={"lat": False, "lon": False, "address_type": True},
    zoom=4.6,                     
    center={"lat": 46.6, "lon": 2.5},   
    height=650,
)

fig.update_layout(
    map_style="open-street-map",  
    margin={"r": 0, "t": 0, "l": 0, "b": 0},
    title="Les 35 destinations Kayak",
)

fig.show()